In [1]:
# 필요한 라이브러리들을 임포트합니다.
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 모델
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트 분류기
from sklearn.model_selection import train_test_split, GridSearchCV # 훈련/테스트 데이터 분할 및 그리드 서치를 위한 모듈
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # 특성 선택을 위한 모듈 (최고 K개 선택, 분산 임계값, ANOVA F-값)
from sklearn.tree import DecisionTreeClassifier # 결정 트리 분류기
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC 점수, F-베타 점수, 커스텀 스코어러 생성
from xgboost import XGBClassifier # XGBoost 분류기
import shap # SHAP(SHapley Additive exPlanations) 라이브러리 (모델 예측 설명)
import matplotlib.pyplot as plt # 데이터 시각화를 위한 라이브러리

import pandas as pd # 데이터 조작 및 분석을 위한 라이브러리
import numpy as np # 수치 계산을 위한 라이브러리
import datetime as dt # 날짜 및 시간 처리를 위한 라이브러리
import json # JSON 데이터 처리를 위한 라이브러리

In [2]:
# 경고 메시지 처리를 위한 모듈
import warnings 

# 'use_label_encoder' 경고만 무시합니다.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [3]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # 파일 경로 지정
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [4]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # 파일 경로 지정
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [5]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)

In [6]:
# 필요한 컬럼만 keep

# 파일 경로 지정
file_path = 'data/cols_to_keep.csv'

# CSV 파일을 DataFrame으로 읽어오기
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]

In [7]:
# data_row <= data_row1 data_row2

# data_row_2에서 조인할 컬럼만 선택
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                  #  'wafer_id', 
                #    'SensorOffsetHot-RoomAfterBake', 
                #    'SensorOffsetHot-ColdAfterBake', 
                   'BG pass/fail']

# 선택한 컬럼으로 data_row_2의 부분집합 DataFrame 생성
data_row_2_subset = data_row_2[columns_to_join]

# data_row_1에 data_row_2의 선택된 컬럼들을 조인 키 'DevID'로 병합
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')

# # 결과 DataFrame 확인
# print(merged_df.head())

In [8]:
initial_dataset = data_row.copy() # 원본 데이터셋 복사
# processed_dataset = initial_dataset.copy() # 원본 데이터셋 복사

In [9]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

In [10]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# 컬럼 이름 변경 딕셔너리 생성
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# .rename() 메서드를 사용하여 컬럼 이름 변경 (inplace=True로 원본 데이터프레임에 바로 적용)
initial_dataset.rename(columns=new_column_names, inplace=True)

# 변경된 컬럼 이름 확인
# print(initial_dataset.columns)

In [11]:
# initial_dataset

In [12]:
# 제거할 컬럼 리스트 정의
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# 컬럼 drop (원본 DataFrame을 변경하려면 inplace=True 사용)
# 또는 새로운 DataFrame을 만들려면 processed_dataset = processed_dataset.drop(...) 사용
initial_dataset.drop(columns=columns_to_drop, inplace=True)

In [13]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

In [14]:
# Radius 컬럼 계산
# np.sqrt() 함수는 각 요소의 제곱근을 계산합니다.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

In [15]:
initial_dataset.to_pickle("data/initial_dataset.p")

#### scnarios

In [16]:
# import vars and function

from algos.algos import *
from config.config import *

In [17]:
##### preprocess_dataset
# def preprocess_dataset(initial_dataset: pd.DataFrame):
    # return processed_dataset # 전처리된 데이터셋 반환

preprocessed_dataset = preprocess_dataset(initial_dataset)

##### create_train_and_test_data
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info

split_parameter = split_parameter_default
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)




     데이터셋 전처리 중...
     전처리 완료!



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

    훈련 및 테스트 데이터셋 생성 중...
    - 분할 전 필터링 미적용.
    - Feature Generation 미적용.

    - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
    - 샘플링 미적용


In [18]:
import copy

In [78]:
# ### senario sample

# ##### train_model_*logistic_regression*
# # def train_model_logistic_regression(train_dataset: pd.DataFrame, feature_selection_info: dict, train_parameters: dict = None):
#     # return model_fitted, importance, model_parameter_info
# trained_model_logistic_regression, feature_importance_logistic_regression, train_parameters_info_logistic_regression = train_model_logistic_regression(train_data, feature_selection_info_var, train_parameters_list_default["random_forest"])

# ##### predict_the_test_data_*logistic_regression*
# # def forecast(test_dataset: pd.DataFrame, trained_model, feature_selection_info: dict):
#     # return predictions, [shap_values, X]
# forecast_dataset_logistic_regression, shap_values_logistic_regression = forecast(test_data, trained_model_logistic_regression, feature_selection_info_var)

# ##### find_best_threshold_*logistic_regression*
# # def find_best_threshold(best_model, train_dataset, feature_selection_info: dict):
#     # return train_dataset, best_threshold
# train_dataset_proba_logistic_regression, best_threshold_logistic_regression = find_best_threshold(trained_model_logistic_regression, train_data, feature_selection_info_var)

# ##### task_roc_logistic_*regression*
# # def roc_from_scratch(probabilities, test_dataset, partitions=100):
#     # return roc_data, auc_score
# roc_data_logistic_regression, auc_score_logistic_regression = roc_from_scratch(forecast_dataset_logistic_regression, test_data, partitions=100)

# ##### create_metrics_on_train_*logistic_regression*
# #def create_metrics_on_train(train_dataset, threshold): 
# #   return train_dataset
# train_dataset_metrics_logistic_regression = create_metrics_on_train(train_dataset_proba_logistic_regression, best_threshold_logistic_regression)

# ##### task_create_metrics_logistic_regression
# # def create_metrics(
# #     predictions: np.array, test_dataset: pd.DataFrame, auc_score, threshold
# # ):
#     # return metrics   
# metrics_logistic_regression = create_metrics(forecast_dataset_logistic_regression, test_data, auc_score_logistic_regression, best_threshold_logistic_regression)

# ##### task_create_results_logistic_regression
# # def create_results(forecast_values, test_dataset, threshold):
#     # return results
# results_logistic_regression = create_results(forecast_dataset_logistic_regression, test_data, best_threshold_logistic_regression)

In [92]:
##### select_feature
# def select_feature(train_data: pd.DataFrame, feature_selector_params: Dict) -> Dict:
    # return feature_selection_info
feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)

feature_selector_params_licor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_licor["filter_methods"]["apply_target_linear_corr_filter"] = True
feature_selector_params_licor["filter_methods"]["target_linear_corr_threshold"] = 0.00
feature_selection_info_licor = select_feature(train_data, feature_selector_params_licor)

feature_selector_params_xicor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_xicor["filter_methods"]["apply_target_xicor_filter"] = True
feature_selector_params_xicor["filter_methods"]["target_xicor_threshold"] = 0.00
feature_selection_info_xicor = select_feature(train_data, feature_selector_params_xicor)

feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "0*median"
feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

# feature_selection_infos = [
#      feature_selection_info_var,
#      feature_selection_info_licor,
#      feature_selection_info_xicor,
#      feature_selection_info_sfm
#      ]

feature_selection_infos = {
     "var" : feature_selection_info_var,
     "licor" : feature_selection_info_licor,
     "xicor" : feature_selection_info_xicor,
     "model" : feature_selection_info_sfm
}


--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 분산 필터링 후 남은 피처 수: 1401

피처 선택 결과가 'data/result/jsons\feature_selection_info_250828_162811_d1ba2bf6.json' 파일에 저장되었습니다.

- 최종 피처 수: 1401

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 선형 상관관계 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250828_162812_cc790198.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 Xi Cor 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250828_162813_e52e940b.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: SFM ---
--- SFM 선택기 완료 ---
남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250828_162815_74e2a7f7.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650


In [93]:
for fileter_name, feature_selection_info in feature_selection_infos.items():
    # print(feature_selection_info["feature_selector_name"])
    # print(feature_selection_info["filter_methods"])
    # print(feature_selection_info["initial_feature_count"])
    # print(feature_selection_info["final_feature_count"])
    # print(feature_selection_info["feature_selection_info_json_path"])

    # print(feature_selection_info["feature_selector_name"])
    print("# of feature:", feature_selection_info["final_feature_count"], ",  filter: ", feature_selection_info["feature_selector_name"], fileter_name)


# of feature: 1401 ,  filter:  FeatureFilter var
# of feature: 1650 ,  filter:  FeatureFilter licor
# of feature: 1650 ,  filter:  FeatureFilter xicor
# of feature: 1650 ,  filter:  SFM model


In [83]:
# feature select test - pipeline

def pl_fs_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results

In [94]:
# feature select test - submit and summary result

ftpn_df = pd.DataFrame()
features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    # ftpn_df에 새로운 행을 추가합니다.
    # pandas.concat을 사용해 행을 추가하는 것이 일반적으로 append보다 효율적입니다.
    ftpn_df = pd.concat([ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    # value_type = ""
    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        # print("y")
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    features_values_dfs = pd.merge(
        features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )

      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
    Best F2 (class=1) score (CV): 0.2258

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7172 with F2 score: 0.6153
      Calculation of the ROC curve...
      Calculation done
      Scoring...
      Scoring done

      Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2240

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.5909
      Calculation of the ROC curve...
      Calculation done
      Sco

In [ ]:
ftpn_df

In [ ]:
features_values_dfs